# Random Lattice Generators Are Not Bad

This notebook explores lattice sequences with random generating vectors,
comparing their integration accuracy against the default (Kuo) generating vectors.

Based on the Python QMCPy demo `lattice_random_generator.ipynb`.

In [ ]:
using QMC
import QMC: Uniform
using Statistics

## Lattice Declaration

Create a lattice with a random generating vector and inspect it.

In [ ]:
# Default lattice (Kuo generating vector)
lat_default = Lattice(2; seed=120)
println("Default lattice:")
println(lat_default)
println()

# Generate samples
n = 16
x = gen_samples(lat_default, n)
println("$n lattice points:")
for i in 1:n
    println("  ", round.(x[i, :], digits=4))
end

## Lattice Point Patterns

Generate lattice points at increasing sample sizes to visualize the extensible structure.

In [ ]:
# Show lattice properties at different sample sizes
let
    for n in [64, 256, 512, 1024]
        lat_n = Lattice(2; seed=136)
        x_n = gen_samples(lat_n, n)
        mn = round.(mean(x_n, dims=1), digits=4)
        sd = round.(std(x_n, dims=1), digits=4)
        println("n=$n: mean=$mn, std=$sd")
    end
end

## Integration Comparison

Compare integration of the Keister function using the default generating vector
vs randomized lattice, both through `CubQMCLatticeG`.

In [ ]:
d = 5
tol = 1e-3

# Default generating vector
dd_default = Lattice(d; seed=7)
tm_default = Gaussian(dd_default; mean=0.0, covariance=0.5)
f_default = Keister(tm_default)
sc_default = CubQMCLatticeG(f_default; abs_tol=tol)
r_default = integrate(sc_default)

println("Default generating vector:")
println("  Solution: $(round(r_default.solution, digits=6))")
println("  Samples:  $(r_default.data[:n])")
println("  Time:     $(round(r_default.data[:time_integrate], digits=4))s")

# Multiple randomized lattices
println("\nRandomized lattice trials:")
for trial in 1:5
    dd_rand = Lattice(d; seed=trial * 100)
    tm_rand = Gaussian(dd_rand; mean=0.0, covariance=0.5)
    f_rand = Keister(tm_rand)
    sc_rand = CubQMCLatticeG(f_rand; abs_tol=tol)
    r_rand = integrate(sc_rand)
    println("  Trial $trial: sol=$(round(r_rand.solution, digits=6)), n=$(r_rand.data[:n])")
end

## Mean vs Median Convergence

Study how the error in the mean-of-means and median-of-means estimators
decay as the sample size increases across multiple replications.

In [ ]:
let
    d = 2
    N_powers = 6:14
    N_list = [2^k for k in N_powers]
    R = 11          # replications per trial
    num_trials = 10

    # Accumulators
    error_mean = zeros(length(N_list))
    error_median = zeros(length(N_list))

    for trial in 1:num_trials
        # Generate R replication estimates at each sample size
        rep_estimates = zeros(R, length(N_list))
        for r in 1:R
            dd_trial = Lattice(d; seed=trial * 1000 + r)
            tm_trial = Gaussian(dd_trial; mean=0.0, covariance=0.5)
            f_trial = Keister(tm_trial)
            x_full = gen_samples(dd_trial, maximum(N_list))
            y_full = evaluate(f_trial, x_full)
            for (j, N) in enumerate(N_list)
                rep_estimates[r, j] = mean(y_full[1:N])
            end
        end
        exact_value = keister_exact(d)
        for (j, _) in enumerate(N_list)
            vals = rep_estimates[:, j]
            error_mean[j] += abs(exact_value - mean(vals))
            error_median[j] += abs(exact_value - median(vals))
        end
    end

    error_mean ./= num_trials
    error_median ./= num_trials

    println("N\t\tMean error\tMedian error")
    for (j, N) in enumerate(N_list)
        println("$(lpad(N, 6))\t$(round(error_mean[j], sigdigits=3))\t\t$(round(error_median[j], sigdigits=3))")
    end
end